In [53]:
customer_water_quality = {
    "客户A": {
        "COD": {"value": 1200, "unit": "mg/L"},
        "氨氮": {"value": 80, "unit": "mg/L"},
        "pH": {"value": 7.2, "unit": None}
    },
    "客户B": {
        "COD": {"value": 3500, "unit": "mg/L"},
        "氨氮": {"value": 150, "unit": "mg/L"},
        "pH": {"value": 6.5, "unit": None}
    }
}

In [54]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,ToolMessage,HumanMessage
from langchain_core.tools import tool
import os

load_dotenv()
api_key = os.getenv('SILICONFLOW_API_KEY')
llm = ChatOpenAI(
        api_key=api_key,
    base_url="https://api.siliconflow.cn/v1",
    model="deepseek-ai/DeepSeek-V3.2"
)


In [55]:
@tool
def get_customer_water_quality(customer_name:str)->dict:
    '''用于查找客户的水质信息'''
    customer_name = customer_name.strip()
    if not customer_name:
        return{
            'sccess':False,
            'data':None,
            'error':'客户信息不能为空'
        }
    if customer_name in customer_water_quality:
        return{
            'success':True,
            'data':customer_water_quality[customer_name],
            'error':None
        }
    return{
            'success':False,
            'data':None,
            'error':f'未找到用户{customer_name}'
        }

In [57]:
get_customer_water_quality.invoke({"customer_name": "客户A"})

{'success': True,
 'data': {'COD': {'value': 1200, 'unit': 'mg/L'},
  '氨氮': {'value': 80, 'unit': 'mg/L'},
  'pH': {'value': 7.2, 'unit': None}},
 'error': None}

In [58]:
get_customer_water_quality.invoke({"customer_name": " 客户A "})

{'success': True,
 'data': {'COD': {'value': 1200, 'unit': 'mg/L'},
  '氨氮': {'value': 80, 'unit': 'mg/L'},
  'pH': {'value': 7.2, 'unit': None}},
 'error': None}

In [59]:
get_customer_water_quality.invoke({"customer_name": ""})

{'sccess': False, 'data': None, 'error': '客户信息不能为空'}

In [60]:
@tool
def calculate_price(volume:float,price:float)->dict:
    '''根据污水的单价和数量计算总价'''
    if volume <= 0 or price <=0:
        return{
            'success':False,
            'data':None,
            'error':'单价和水量必须大于0'
        }
    return{
        'success':True,
        'data':{
            "volume":volume,
            "price":price,
            "total_price":volume*price
        },
        'error':None
    }

In [61]:
calculate_price.invoke({
    "volume": 40,
    "price": 30
})

{'success': True,
 'data': {'volume': 40.0, 'price': 30.0, 'total_price': 1200.0},
 'error': None}

In [62]:
calculate_price.invoke({
    "volume": 40,
    "price": -5
})

{'success': False, 'data': None, 'error': '单价和水量必须大于0'}

In [63]:
def check_relevance(query:str,context:str,llm)->bool:
    prompts = f'''
    请判断下面提供的企业内部资料是否能够回答用户问题。

用户问题：
{query}

检索资料：
{context}

判断规则：
- 如果资料中存在能够直接支持回答问题的信息，只回答 YES
- 如果资料与问题无关，或不足以回答，只回答 NO
- 不要输出其他内容
    '''
    response = llm.invoke(prompts)
    return response.content.strip().upper() == "YES"

In [64]:
import os 
os.chdir('..')
from main import initial_rag
from src.query_processing import rewrite_query
from src.retrieval import bm25_search,faiss_search
from src.fusion import rrf_fusion
from src.rag import build_context
bm25, vectorstore, documents,llm = initial_rag()
from langchain_core.tools import tool

@tool
def knowledge_search(query:str)->dict:
    """查询企业污水业务制度、SOP、合同流程等内部知识。"""
    if not query:
        return{
            'success':False,
            'data':None,
            'error':'问题不能为空'
        }
    query = query.strip()
    rewritten_query = rewrite_query(query,llm)
    bm25_result = bm25_search(rewritten_query,bm25,documents,k=5)
    faiss_result = faiss_search(rewritten_query,vectorstore,k=5)
    rrf_results = rrf_fusion(bm25_result,faiss_result,top_k=3)
    context = build_context(rrf_results,top_k=3)
    is_relevance = check_relevance(query,context,llm)
    if not is_relevance:
        return{
            'success':False,
            'data':None,
            'error':'未检索到足够相关的企业内部资料'
        }
    return{
        'success':True,
        'data' : context,
        'error': None
    }

索引及hash有效，加载FAISS向量存储。
FAISS对象类型：<class 'langchain_community.vectorstores.faiss.FAISS'>


In [65]:
import inspect
print('rewrite',inspect.signature(rewrite_query))
print('bm225search',inspect.signature(bm25_search))
print('faiss',inspect.signature(faiss_search))
print('rrf',inspect.signature(rrf_fusion))
print('build_context',inspect.signature(build_context))

rewrite (query, llm)
bm225search (query, bm25, documents, k=5)
faiss (query, vectorstore, k=5)
rrf (bm25_results, faiss_results, k=60, top_k=None, bm25_weight=0.5, faiss_weight=1.0)
build_context (rrf_results, top_k=3)


In [66]:
knowledge_search.invoke({
    "query": "外委污水合同签订前需要做哪些准备？"
})

{'success': True,
 'data': '来源：未知来源|F002_C012\n内容：Section: 4  管理事项与工作程序\nSubsection: 4.2  接收流程\nContent: 4  管理事项与工作程序\n4.2  接收流程\n\n4.2.1  前期对接\n发展与市场部负责收集客户信息，由客户填写《燕山威立雅外委污水新增客户信息表及承诺书》（附件6.3），并对客户的营业执照、排污许可证、开户许可证等资质信息进行审核。\n4.2.2  可行性评估\n客户资质审核通过后，发展与市场部填写《外委污水处理业务评估单》（附件6.2）中“外委污水处理相关信息”等内容，由生产部负责对外委污水接纳的工艺处理的可行性进行评估，包括但不限于是否接收、处理方案、接收地点及接收量等内容；安全环保部负责对环保合规性进行评估，给出是否属于危险废物的评估意见。\n4.2.3  合同谈判及签订\n发展与市场部负责与客户进行接洽、谈判，财务部配合完成成本测算，经理办公室对合同法律合规性进行审核。发展与市场部按照内控流程签订《外委污水处理业务服务协议》，填写《北京燕山威立雅水务有限责任公司新增供应商审批单》，向财务部提出新增客户申请。\n4.2.4  接收及合规达标处理\n生产部负责对已完成《外委污水处理业务服务协议》签订工作的污水接收及达标处理进行管控。各生产单位按生产部要求执行外委污水的接收及达标处理，接收过程中若发现水质异常，须第一时间开展应急处置，并即刻向生产部及公司管理层上报。\n4.2.5  费用结算\n由发展与市场部负责发起费用结算申请，财务部负责进行结算。\n\n----------\n\n来源：未知来源|F010_C013\n内容：Section: 八、业务流程\nSubsection: 8.4内部评估\nContent: 八、业务流程\n8.4内部评估\n\n目的\n\n根据水质情况，客户的现场情况、资质等相关情况，联合公司相关部门判定是否接该水样。\n\n**8.4.1 Input**\n\n1. 客户水质指标。\n2. 非常明确的客户的情况：污水性质、需处理水量等。\n\n**8.4.3 内部评估的一般流程**\n\n1. 和客户公司联系获取营业执照、排污许可证，确定客户公司是可以合法转运污水的。\n2. 填写内部燕山威立雅外委污水处理业务评估单

In [67]:
knowledge_search.invoke({
    "query": "火星人的工作时间是什么时候？"
})

{'success': False, 'data': None, 'error': '未检索到足够相关的企业内部资料'}

In [70]:
tools = [get_customer_water_quality,calculate_price,knowledge_search]
tool_map = {tool.name:tool
           for tool in tools}
llm_with_tool = llm.bind_tools(tools)
def agent_loop(question:str,max_step=5):
    message = [SystemMessage(content=
                            '''
                            你是一个污水接收业务助手，请你根据制度、sop等有的你文件进行回复'''),
              HumanMessage(content=question)]

    for step in range(max_step):
        response = llm_with_tool.invoke(message)
        message.append(response)
        if not response.tool_calls:
            return response.content
        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call['id']
            tool_result = tool_map[tool_name].invoke(tool_args)
            print("当前轮次:", step)
            print("调用工具:", tool_name)
            print("调用参数:", tool_args)
            print("工具结果:", tool_result)
            message.append(
                ToolMessage(content=str(tool_result),
                           tool_call_id=tool_id)
            )
    return f'超出最大循环次数，此循环直接结束'

In [71]:
test_questions = [
    "40吨污水，每吨30元，一共多少钱？",
    "客户A的水质数据是什么？",
    "客户C的水质数据是什么？",
    "外委污水合同签订前需要做哪些准备？",
    "火星人的工作制度是什么？",
    "客户A有40吨污水，单价30元，请告诉我水质数据和总处理费用。",
    "客户B有20吨污水，单价50元，请告诉我水质和费用。",
    "0吨污水，每吨30元，一共多少钱？"
]
for question in test_questions:
    print("=" * 60)
    print("问题：", question)
    answer = agent_loop(question)
    print("回答：", answer)

问题： 40吨污水，每吨30元，一共多少钱？
当前轮次: 0
调用工具: calculate_price
调用参数: {'volume': 40, 'price': 30}
工具结果: {'success': True, 'data': {'volume': 40.0, 'price': 30.0, 'total_price': 1200.0}, 'error': None}
回答： 40吨污水，按每吨30元计算：

**总价 = 40 × 30 = 1,200元**

如需按制度办理后续接收、签约或结算流程，可以告诉我客户名称等信息，我再帮您查询相关规定和客户水质情况。
问题： 客户A的水质数据是什么？
当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户A'}
工具结果: {'success': True, 'data': {'COD': {'value': 1200, 'unit': 'mg/L'}, '氨氮': {'value': 80, 'unit': 'mg/L'}, 'pH': {'value': 7.2, 'unit': None}}, 'error': None}
回答： 客户A的水质数据如下：

| 指标 | 数值 | 单位 |
|------|------|------|
| COD（化学需氧量） | 1200 | mg/L |
| 氨氮 | 80 | mg/L |
| pH | 7.2 | — |

如需进一步判断该水质是否满足接收标准、核算处理价格，或查询相关制度与SOP，请告诉我。
问题： 客户C的水质数据是什么？
当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户C'}
工具结果: {'success': False, 'data': None, 'error': '未找到用户客户C'}
回答： 我查询了系统，**未找到“客户C”的水质数据**，系统返回：`未找到用户客户C`。

可能的原因：
- 客户名称不完整或不准确（系统里可能登记的是客户全称、简称或客户编号）；
- 该客户尚未建档，或还未录入水质信息。

请您提供**客户的完整名称或客户编号**，我再帮您查询。


In [3]:
import os 
from pathlib import Path
import pandas as pd
file_path = Path.cwd().parent/'data'/'structured'/'price_history.xlsx'
df = pd.read_excel(file_path)

In [4]:
print(df.head())

  customer_name           sample_date case_type case_status  water_volume  \
0           生态岛  2026.8.14-2026.10.29    稳定长期合作         已处理        3000.0   
1          丰泰民安              2026.7.9     未成交案例         未处理         200.0   
2          集联光电              2026.8.9     合同协商中         已处理           NaN   
3          天宇玉慧                2026.7    稳定长期合作         未处理           NaN   
4          天宇玉慧              2026.9.8      异常调价         已处理         200.0   

             volume_note     PH     COD   CODS     BOD5  ...   TOC   氟化物  \
0   累计已处理约3000吨，目前约30吨/天  10.23     370      /    135.0  ...   NaN   NaN   
1  事故存水约200吨，恢复生产后约30吨/天   3.55  184264  76529  43000.0  ...   NaN   NaN   
2          日送水量约30–50吨/天   7.19     195    158     85.6  ...  31.1  1.25   
3          单次约200吨，频次不固定    6-9    1515      /    763.0  ...     /     /   
4                单次约200吨   6.84    8916      /   3630.0  ...   NaN   NaN   

     TDS  TDS_min  TDS_max initial_quote actual_settlement_price  \
0  25200    

In [5]:
print(df.columns.tolist())

['customer_name', 'sample_date', 'case_type', 'case_status', 'water_volume', 'volume_note', 'PH', 'COD', 'CODS', 'BOD5', 'NH3_N', 'TN', 'TP', 'SS', 'OIL', 'TOC', '氟化物', 'TDS', 'TDS_min', 'TDS_max', 'initial_quote', 'actual_settlement_price', 'contract_price', 'transfer', 'remarks']


In [6]:
print(df.dtypes)

customer_name               object
sample_date                 object
case_type                   object
case_status                 object
water_volume               float64
volume_note                 object
PH                          object
COD                          int64
CODS                        object
BOD5                       float64
NH3_N                      float64
TN                         float64
TP                         float64
SS                           int64
OIL                        float64
TOC                         object
氟化物                         object
TDS                          int64
TDS_min                     object
TDS_max                     object
initial_quote              float64
actual_settlement_price    float64
contract_price             float64
transfer                    object
remarks                     object
dtype: object


In [9]:
import numpy as np

df = df.replace("/", np.nan)

In [10]:
print(df.head())

  customer_name           sample_date case_type case_status  water_volume  \
0           生态岛  2026.8.14-2026.10.29    稳定长期合作         已处理        3000.0   
1          丰泰民安              2026.7.9     未成交案例         未处理         200.0   
2          集联光电              2026.8.9     合同协商中         已处理           NaN   
3          天宇玉慧                2026.7    稳定长期合作         未处理           NaN   
4          天宇玉慧              2026.9.8      异常调价         已处理         200.0   

             volume_note     PH     COD     CODS     BOD5  ...   TOC   氟化物  \
0   累计已处理约3000吨，目前约30吨/天  10.23     370      NaN    135.0  ...   NaN   NaN   
1  事故存水约200吨，恢复生产后约30吨/天   3.55  184264  76529.0  43000.0  ...   NaN   NaN   
2          日送水量约30–50吨/天   7.19     195    158.0     85.6  ...  31.1  1.25   
3          单次约200吨，频次不固定    6-9    1515      NaN    763.0  ...   NaN   NaN   
4                单次约200吨   6.84    8916      NaN   3630.0  ...   NaN   NaN   

     TDS  TDS_min  TDS_max  initial_quote  actual_settlement_price  

In [11]:
df.loc[df['PH']=='6-9','PH'] = np.nan


In [12]:
water_quality_cols = [
    "PH",
    "COD",
    "CODS",
    "BOD5",
    "NH3_N",
    "TN",
    "TP",
    "SS",
    "OIL",
    "TOC",
    "氟化物",
    "TDS",
    "TDS_min",
    "TDS_max"
]

In [13]:
for col in water_quality_cols:
    df[col] = pd.to_numeric(df[col],errors='coerce')

In [16]:
print(df.dtypes)

customer_name               object
sample_date                 object
case_type                   object
case_status                 object
water_volume               float64
volume_note                 object
PH                         float64
COD                          int64
CODS                       float64
BOD5                       float64
NH3_N                      float64
TN                         float64
TP                         float64
SS                           int64
OIL                        float64
TOC                        float64
氟化物                        float64
TDS                          int64
TDS_min                    float64
TDS_max                    float64
initial_quote              float64
actual_settlement_price    float64
contract_price             float64
transfer                    object
remarks                     object
dtype: object


In [19]:
standards = {
    "COD": 600,
    "CODS": 300,
    "SS": 200,
    "NH3_N": 30,
    "TN": 70,
    "TP": 3.2,
    "TDS": 1600,
    "BOD5": 250,
    "TOC": 150,
    "OIL": 30
}
base_price = 8.93

In [18]:
row = df.iloc[1]

cod_ratio = row["COD"] / standards["COD"]
tds_ratio = row["TDS"] / standards["TDS"]

print("COD倍数:", cod_ratio)
print("TDS倍数:", tds_ratio)

COD倍数: 307.1066666666667
TDS倍数: 7.20625


这周目标我给你重新收成一句话：

> **把现在这个“能跑的 RAG+Agent Demo”，升级成一个有真实业务数据、有报价辅助能力、有基础评估、能工程化展示、能写进简历的 Agent V0.2。**

本周不要再乱扩技术栈，主线就 5 件事：

1. **做出报价辅助 Tool**
   - 规则价：最高超标倍数 × 8.93
   - 返回最大超标指标、倍数、规则参考价
   - 查询历史案例
   - 返回历史报价/合同价/实际结算价/备注
   - 最终只给参考建议，不自动拍板

2. **把真实业务数据接进 Agent**
   - 不再只用 mock 数据
   - 至少接入你现在这 7 条真实 case
   - 水质、报价、合同、结算信息能被 Tool 使用

3. **把 Agent 跑成多 Tool 场景**
   - 知识检索
   - 客户水质查询
   - 报价辅助
   - 费用计算
   - 能根据问题自己选择必要 Tool

4. **做一轮正式 Evaluation**
   - 现在 8 题扩到大约 15～20 题
   - 覆盖单 Tool、多 Tool、异常、无关问题、过度调用
   - 至少保留 Tool Routing Accuracy 和失败案例

5. **工程化并形成可展示版本**
   - `src/tools.py`
   - `src/agent.py`
   - FastAPI/Streamlit 能跑
   - README 写清：业务痛点、难点、架构、指标、价值
   - 周末能用 5 分钟讲完整项目

这周**先不做**：
- SFT / DPO / GRPO
- K8s 深挖
- MCP 深挖
- 多 Agent
- 复杂 Memory
- 为了炫技做复杂相似度算法

你明天最优先的任务非常明确：

> **先把报价辅助 Tool V0.1 做出来。**

明天完成这个，就算本周最关键的一块真正落地了。

好，那从现在开始我就按“参谋长”模式来管进度，不再用“今天也还行”这种安慰性评价。

你的总目标只有一个：

> **2027 春节前，尽最大概率拿到 AI 售前工程师 / AI 解决方案工程师的正式 offer。**

所有学习、项目、资料整理、投递，都必须服务这个目标。做不到这一点的任务，我会主动砍掉。

我给你设三层里程碑：

| 层级 | 截止时间 | 必须拿到的结果 |
|---|---|---|
| **本周战役：Agent V0.2** | **11/15 前** | 报价辅助 Tool + Agent 接入 + 15～20题评估 + 工程化 + 可演示 |
| **公司级里程碑** | **11月底前争取触发** | 拿着能演示的 PoC 去申请公司大模型 / Embedding / Rerank / 服务器资源 |
| **求职里程碑** | **12月进入系统投递，春节前冲 offer** | 项目、简历、方案能力、面试表达全部进入实战 |

而且从明天开始，本周我会按**硬验收**来卡你。

**11月10日必须完成：** `get_price_reference()` V0.1。至少能输入水质，计算规则参考价，返回最大超标指标、倍数、8.93 基础价计算结果，再返回历史报价案例。至少拿 3 个真实 case 测通。这个明天做不完，就是明确落后，不找理由。

**11月11日必须完成：** 报价 Tool 接进 Agent。能够问类似“这个水质按规则多少钱，历史有没有参考案例”，Agent 自己调用正确 Tool，并给出受控的辅助建议。

**11月12日必须完成：** 测试集从现在 8 题扩到 15～20 题，覆盖知识查询、水质、报价、计算、多 Tool、失败场景、无关问题。Tool Routing 和失败案例有结果。

**11月13日必须完成：** Notebook 里的稳定代码迁进 `src/tools.py`、`src/agent.py`，重新跑回归。这个确实不应该拖一天，属于顺手收口。

**11月14日必须完成：** 接回 FastAPI + Streamlit，浏览器里能完整演示一个真实业务问题，而不是 Notebook Demo。

**11月15日必须完成：** README、架构图、项目难点、业务收益、评估指标、5分钟项目讲解全部形成。到这一天，我会明确判断：这个项目能不能进入简历、能不能拿给公司领导看、还差什么。

然后第二阶段不是继续无休止开发。

**11月下半月重点会转向两件事：一是公司内部 AI 资源权限；二是求职材料。**

服务器权限我不会让你“等项目完美”。触发条件非常明确：

> PoC 能稳定演示 + 能说明业务价值 + 能说明为什么需要公司模型/Embedding/Rerank资源。

一旦满足，我们就开始准备你找总经理时的那一页材料：**现在有什么、解决什么问题、为什么需要资源、需要什么资源、拿到后准备验证什么。**

求职这条线也一样。我不会让你学到明年再投。到 11 月底、12 月初，如果项目达到可讲程度，就必须开始更系统地投。之后每周都要有：

```text
项目能力增长
+
JD要求校准
+
针对性投递
+
面试复盘
```

并行推进。

每天晚上 21:50 的打卡，我以后不只是问“今天学了什么”。我会按三个问题直接判：

> **今天有没有完成计划产出？  
> 有没有影响本周里程碑？  
> 明天要不要补进度？**

如果当天慢了，我会直接告诉你“落后”，并压缩第二天低优先级任务补回来；如果你又开始研究一个与当前目标无关的新技术，我会直接让你停。

今天就是一个例子：**今天进度确实偏慢。**资料整理有价值，但本周核心工程产出没有达到原计划。所以明天不能继续整理资料发散，第一优先级就是把报价 Tool 打穿。

从现在开始，我们不追求“每天感觉学了不少”，只追求：

> **每周必须跨过一个能提高你拿 offer 概率的门槛。**

这个尺度我帮你卡。